# Demo: HEP-multiagent + kb-mcp


## 0. Setup


In [ ]:
%pip install -q -U uv langchain-openai
%pip install -q "git+ssh://git@github.com/HEP-KE/HEP-multiagent.git"


## 1. Argo LLM


In [ ]:
import os
from langchain_openai import ChatOpenAI

os.environ["ARGO_BASE_URL"] = "https://apps-dev.inside.anl.gov/argoapi/v1"
os.environ["ARGO_MODEL"] = "GPT-5.5"
os.environ["ARGO_USER"] = "YOUR_ARGO_USERNAME"

if os.environ["ARGO_USER"] == "YOUR_ARGO_USERNAME":
    raise ValueError("Set ARGO_USER to your Argo username before running this cell.")

llm = ChatOpenAI(
    model=os.environ["ARGO_MODEL"],
    base_url=os.environ["ARGO_BASE_URL"],
    api_key=os.environ["ARGO_USER"],
)


## 2. Tiny KB


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

kb_root = Path("demo_kb_runtime").resolve()
data_dir = kb_root / "data"
db_path = kb_root / "kb.sqlite"
note_path = kb_root / "detector_note.txt"
uv_tool_args = ["-m", "uv", "tool", "run", "--from", "git+ssh://git@github.com/HEP-KE/kb-mcp.git"]

data_dir.mkdir(parents=True, exist_ok=True)
if not note_path.exists():
    note_path.write_text(
        "Detector timing cuts and tracker resolution improve sensitivity "
        "while reducing beam background and readout noise.\n"
    )

kb_env = os.environ.copy()
for name in ("DB_HOST", "DB_PORT", "DB_USER", "DB_PASSWORD", "DB_NAME", "DB_SCHEMA", "DB_URL"):
    kb_env.pop(name, None)
kb_env.update({"DB_USER": "", "DB_NAME": "", "SQLITE_DB_PATH": str(db_path), "DATA_DIR": str(data_dir)})

if not db_path.exists():
    subprocess.run(
        [
            sys.executable, *uv_tool_args,
            "kb", "ingest", str(note_path),
            "--source-id", "demo",
            "--doc-id", "detector-note",
            "--batch",
            "--no-summary",
            "--no-summary-chunks",
            "--no-embed",
            "--no-copy",
        ],
        env=kb_env,
        check=True,
    )

print(db_path)


## 3. Initialize Agent


In [ ]:
from hep_multiagent import Agent, AgentFeatures

mcp_servers = [{
    "name": "kb-mcp",
    "transport": "stdio",
    "command": sys.executable,
    "args": [*uv_tool_args, "kb-server-stdio"],
    "env": kb_env,
}]

agent = Agent(
    llm=llm,
    mcp_servers=mcp_servers,
    features=AgentFeatures(
        report=True,
        citations=False,
        replay_notebook=True,
        execution_log=True,
        structured_worker_output=True,
    ),
)


## 4. Run


In [ ]:
result = await agent.run(
    "Use kb_get with identifier demo_detector-note. "
    "Summarize the detector note in two short bullet points. Do not call kb_search.",
    output_dir=str(kb_root / "agent_output"),
)

print(result.get("final_report", result))
